# 評価と分析

Time-to-Warn指標、Precision@K、False Positive分析、概念ドリフト評価を行う。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# パス設定
if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive')
    project_dir = Path('/content/drive/MyDrive/evil_package')
    sys.path.insert(0, str(project_dir))
else:
    project_dir = Path('..')

import os
os.chdir(project_dir)

In [ ]:
from src.evaluation.metrics import EvaluationMetrics
import pickle
import json

## モデルとデータの読み込み

学習済みモデルとテストデータを読み込む。

In [ ]:
# モデルを読み込み
with open('results/models/anomaly_detector.pkl', 'rb') as f:
    model_data = pickle.load(f)

detector = model_data['model']
threshold = model_data['threshold']

# テストデータを読み込み
test_data = np.load("data/processed/features_test.npz")
X_nlp_test = test_data['X_nlp']
X_metadata_test = test_data['X_metadata']
y_test = test_data['y']
names_test = test_data['names']

# 元のデータから公開時刻を取得
with open("data/splits/test.json", "r") as f:
    test_raw_data = json.load(f)

# 公開時刻とパッケージ名のマッピング
publication_times = []
for name in names_test:
    for data in test_raw_data:
        if data['package_name'] == name:
            created_str = data['metadata'].get('created', '')
            if created_str:
                try:
                    pub_time = datetime.fromisoformat(created_str.replace('Z', '+00:00'))
                    publication_times.append(pub_time)
                    break
                except:
                    publication_times.append(None)
                    break
    else:
        publication_times.append(None)

publication_times = np.array(publication_times)
print(f"Loaded {len(y_test)} test samples")

## Time-to-Warn指標の計算

公開から検知までの時間を分析する。

In [ ]:
# 予測
test_scores = detector.predict_proba(X_nlp_test, X_metadata_test)
test_pred = detector.predict(X_nlp_test, X_metadata_test, threshold=threshold)
test_pred_binary = (test_pred == -1).astype(int)

# 検知時刻をシミュレート（実際には検知時刻を記録する必要がある）
# ここでは公開時刻から一定時間後と仮定
detection_times = []
for i, pub_time in enumerate(publication_times):
    if pub_time and test_pred_binary[i] == 1:
        # 検知時刻 = 公開時刻 + スコアに基づく遅延（簡易版）
        delay_hours = test_scores[i] * 24  # スコアが高いほど早く検知
        from datetime import timedelta
        det_time = pub_time + timedelta(hours=delay_hours)
        detection_times.append(det_time)
    else:
        detection_times.append(None)

detection_times = np.array(detection_times)

# Time-to-Warn指標を計算
metrics = EvaluationMetrics()

# 有効な時刻データのみを使用
valid_mask = np.array([pt is not None and dt is not None for pt, dt in zip(publication_times, detection_times)])

if np.any(valid_mask):
    time_metrics = metrics.compute_time_to_warn(
        y_test[valid_mask],
        test_pred_binary[valid_mask],
        publication_times[valid_mask],
        detection_times[valid_mask],
        time_windows=[60, 360, 1440]  # 1時間、6時間、24時間
    )
    
    print("Time-to-Warn Metrics:")
    for key, value in time_metrics.items():
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")

## Precision@Kの計算

人間が1日で見れる件数での精度を評価する。

In [ ]:
# Precision@Kを計算
precision_at_k = metrics.compute_precision_at_k(y_test, test_scores, k_values=[10, 50, 100])

print("Precision@K:")
for k, precision in precision_at_k.items():
    print(f"{k}: {precision:.4f}")

# 可視化
plt.figure(figsize=(8, 5))
k_values = [10, 50, 100]
precisions = [precision_at_k[f'precision_at_{k}'] for k in k_values]
plt.plot(k_values, precisions, marker='o')
plt.xlabel('K (Top K packages)')
plt.ylabel('Precision')
plt.title('Precision@K')
plt.grid(True)
plt.savefig('results/figures/precision_at_k.png')
plt.show()

## False Positive分析

誤検知の原因を分析する。

In [ ]:
# False Positive分析
fp_analysis = metrics.analyze_false_positives(y_test, test_pred_binary, test_scores)

print("False Positive Analysis:")
for key, value in fp_analysis.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

# False Positiveのパッケージ名を表示
fp_mask = (y_test == 0) & (test_pred_binary == 1)
fp_packages = names_test[fp_mask]
fp_scores = test_scores[fp_mask]

print(f"\nFalse Positive packages (top 10 by score):")
fp_df = pd.DataFrame({
    'package_name': fp_packages,
    'score': fp_scores
}).sort_values('score', ascending=False)
print(fp_df.head(10))

## 概念ドリフト評価

時系列での性能劣化を評価する。

In [ ]:
# 時系列ラベルを作成（年ごと）
time_periods = []
for pub_time in publication_times:
    if pub_time:
        time_periods.append(pub_time.year)
    else:
        time_periods.append(2024)  # デフォルト

time_periods = np.array(time_periods)

# 時系列での性能を計算
temporal_performance = metrics.compute_temporal_performance(
    y_test, test_pred_binary, test_scores, time_periods
)

print("Temporal Performance:")
print(temporal_performance)

# 可視化
if len(temporal_performance) > 0:
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(temporal_performance['period'], temporal_performance['precision'], marker='o', label='Precision')
    plt.plot(temporal_performance['period'], temporal_performance['recall'], marker='s', label='Recall')
    plt.xlabel('Year')
    plt.ylabel('Score')
    plt.title('Performance Over Time')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(temporal_performance['period'], temporal_performance['auc_roc'], marker='o', label='AUC-ROC')
    plt.xlabel('Year')
    plt.ylabel('AUC-ROC')
    plt.title('AUC-ROC Over Time')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig('results/figures/temporal_performance.png')
    plt.show()

## 評価レポートの生成

すべての評価結果をまとめたレポートを生成する。

In [ ]:
# 全評価指標を計算
all_metrics = metrics.compute_all_metrics(
    y_test, test_pred_binary, test_scores,
    publication_times if np.any(valid_mask) else None,
    detection_times if np.any(valid_mask) else None
)

# レポートを保存
report = {
    'evaluation_date': datetime.now().isoformat(),
    'metrics': all_metrics,
    'precision_at_k': precision_at_k,
    'false_positive_analysis': fp_analysis,
    'temporal_performance': temporal_performance.to_dict('records') if len(temporal_performance) > 0 else []
}

with open('results/reports/evaluation_report.json', 'w') as f:
    json.dump(report, f, indent=2, default=str)

print("Evaluation report saved successfully")